# Language Modeling with Neural Networks

Adapted from: https://github.com/BrownFortress/NLU-2024-Labs

## 1. Data preprocessing

This section is devoted to process the data in order to make it compatible with the neural network framework.

To train and evaluate the LM we'll use the Peen TreeBank (PTB) corpus composed by text taken from articles of the Wall Street Journal with around 1.2 million tokens in total. We'll load train, validation and test datasets ($\mathcal{D}_{train}$, $\mathcal{D}_{val}$, $\mathcal{D}_{test}$) and convert tokens into ids, then define the dataloader that control return the batches to send to the neural network.

In [5]:
# Run these lines only once to download the data
!wget -P dataset/PennTreeBank https://raw.githubusercontent.com/BrownFortress/NLU-2024-Labs/main/labs/dataset/PennTreeBank/ptb.test.txt
!wget -P dataset/PennTreeBank https://raw.githubusercontent.com/BrownFortress/NLU-2024-Labs/main/labs/dataset/PennTreeBank/ptb.valid.txt
!wget -P dataset/PennTreeBank https://raw.githubusercontent.com/BrownFortress/NLU-2024-Labs/main/labs/dataset/PennTreeBank/ptb.train.txt

--2024-12-18 11:09:52--  https://raw.githubusercontent.com/BrownFortress/NLU-2024-Labs/main/labs/dataset/PennTreeBank/ptb.test.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 449945 (439K) [text/plain]
Saving to: ‘dataset/PennTreeBank/ptb.test.txt.1’

ptb.test.txt.1      100%[===================>] 439.40K  --.-KB/s    in 0.03s   

2024-12-18 11:09:52 (15.0 MB/s) - ‘dataset/PennTreeBank/ptb.test.txt.1’ saved [449945/449945]

--2024-12-18 11:09:52--  https://raw.githubusercontent.com/BrownFortress/NLU-2024-Labs/main/labs/dataset/PennTreeBank/ptb.valid.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.13

In [7]:
# Load a corpus
def read_file(path, eos_token="<eos>"):
    output = []
    with open(path, "r") as f:
        for line in f.readlines():
            output.append(line.strip() + " " + eos_token)
    return output

# Load train, validation and test corpus
train_raw = read_file("dataset/PennTreeBank/ptb.train.txt")
dev_raw = read_file("dataset/PennTreeBank/ptb.valid.txt")
test_raw = read_file("dataset/PennTreeBank/ptb.test.txt")

The Lang class control the conversion from token to id and viceversa, it computes and stores a vocabolary of the words available in the dataset.

In [8]:
# Language vocabolary class
class Lang():
    def __init__(self, corpus, special_tokens=[]):

        # Vocab dict for word to id and reverse
        self.word2id = self.get_vocab(corpus, special_tokens)
        self.id2word = {v:k for k, v in self.word2id.items()}

    # Returns vocab dict for word to ids given a corpus
    def get_vocab(self, corpus, special_tokens=[]):
        output = {}
        i = 0
        # Add special words [<eos>, <pad>] to the vocab
        for st in special_tokens:
            output[st] = i
            i += 1
        # Add new words found in the corpus to the vocab
        for sentence in corpus:
            for w in sentence.split():
                if w not in output:
                    output[w] = i
                    i += 1
        return output

In [4]:
# Compute the vocabolary from the dataset
# "<pad>" will be used to fill shorter sentences
# "<eos>" is put at the end of the sentence
SPECIAL_TOKENS = ["<pad>", "<eos>"]
lang = Lang(train_raw, SPECIAL_TOKENS)

# Vocabolary size
print(len(lang.word2id))

10001


PyTorch really helps you to manage your datasets with built in classes, here we use the Dataset class to store our data and sequentially convert:

- tokens $\to$ ids
- ids $\to$ Torch tensors

In [9]:
import torch
import torch.utils.data as data

# Dataset torch class
class PennTreeBank (data.Dataset):
    # Mandatory methods are __init__, __len__ and __getitem__
    def __init__(self, corpus, lang):

        # List of input (source) and ground truth (target) sequences
        self.source = []
        self.target = []

        # Foreach sentence in the corpus extract input and output sequence (self supervised learning)
        for sentence in corpus:
            self.source.append(sentence.split()[0:-1])
            self.target.append(sentence.split()[1:])

        # Convert sequences of tokens into sequences of ids
        self.source_ids = self.mapping_seq(self.source, lang)
        self.target_ids = self.mapping_seq(self.target, lang)

    def __len__(self):
        return len(self.source)

    # Return a sample
    def __getitem__(self, idx):
        src= torch.LongTensor(self.source_ids[idx])
        trg = torch.LongTensor(self.target_ids[idx])
        sample = {'source': src, 'target': trg}
        return sample

    # Auxiliary methods

    # Map sequences of tokens to corresponding ids computed in Lang class
    def mapping_seq(self, data, lang):
        res = []
        # Foreach sequence in the data
        for seq in data:
            tmp_seq = []
            # Foreach token in the sentence
            for x in seq:
                # If the token is in the vocab convert it, otherwise OOV (Out Of Vocabolary) token
                if x in lang.word2id:
                    tmp_seq.append(lang.word2id[x])
                else:
                    # PennTreeBank doesn't have OOV but "Trust is good, control is better!"
                    print('OOV found!')
                    print('You have to deal with that')
                    print(f'Token OOV: {x}')
                    break
            res.append(tmp_seq)
        return res

In [11]:
# Get datasets converting tokens into torch tensor with ids
train_dataset = PennTreeBank(train_raw, lang)
dev_dataset = PennTreeBank(dev_raw, lang)
test_dataset = PennTreeBank(test_raw, lang)

Neural networks can easily make the most of the parallel computing techniques made available from modern hardware (ex. GPUs) to process multiple examples at the same time (this is also one of the reasons of their success).

Instead of sending a single training pair $\{\mathbf{x}_i,\mathbf{y}_i\}$ to the network we send a batch of them $\mathcal{B} = \{\{\mathbf{x}_1,\mathbf{y}_1\},\dots,\{\mathbf{x}_b,\mathbf{y}_b\}\}$. Also here PyTorch help us with the DataLoader class to obtain batches from the Dataset class.

In [12]:
# Assign correct device wrt. hardware possibilities
# Check GPU availability
if torch.cuda.is_available():
    DEVICE = 'cuda:0'
else:
    DEVICE = 'cpu'

In [14]:
# Compute batches sampling a list of samples
def collate_fn(data, pad_token):

    def merge(sequences):
        '''
        merge from batch * sent_len to batch * max_len
        '''
        lengths = [len(seq) for seq in sequences]
        max_len = 1 if max(lengths)==0 else max(lengths)
        # Create a matrix full of PAD token (i.e. 0) with the shape (batch_size, maximum length of a sequence)
        # copying the sequence until its end and filling with PAD token
        padded_seqs = torch.LongTensor(len(sequences),max_len).fill_(pad_token)
         # Copy each sequence into the matrix
        for i, seq in enumerate(sequences):
            end = lengths[i]
            padded_seqs[i, :end] = seq
        # Remove these tensors from the computational graph
        padded_seqs = padded_seqs.detach()
        return padded_seqs, lengths


    # Sort data by seq lengths
    data.sort(key=lambda x: len(x["source"]), reverse=True)
    # Get dict of lists (input_seq, ground_truth_sequence) from list of samples
    new_item = {}
    for key in data[0].keys():
        new_item[key] = [d[key] for d in data]

    # Load the tensors on our selected device
    source, _ = merge(new_item["source"])
    target, lengths = merge(new_item["target"])

    # Return item required to compose the batch
    new_item["source"] = source.to(DEVICE)
    new_item["target"] = target.to(DEVICE)
    new_item["number_tokens"] = sum(lengths)
    return new_item

In [15]:
from torch.utils.data import DataLoader
from functools import partial

# Batch size
BATCH_SIZE = 64

# Dataloader instantiation
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, collate_fn=partial(collate_fn, pad_token=lang.word2id["<pad>"]), shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE, collate_fn=partial(collate_fn, pad_token=lang.word2id["<pad>"]))
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, collate_fn=partial(collate_fn, pad_token=lang.word2id["<pad>"]))

## 2. Recurrent Neural Networks for LM

I implemented for you by hand the RNN to make you understand how it works, but we'll use the one that PyTorch provides to help us speed up the process (don't plug my version it in the training code because it doesn't work).

In addiction you can find a better model, I suggest you to use that one if you want to generate better text (don't expect it to be at GPT level of course).

I leave here also the original paper of LM with RNN from Mikolov et al. (https://www.isca-archive.org/interspeech_2010/mikolov10_interspeech.pdf) if you want to see if you can beat its performance later. It doesn't count if you beat him using my model :).

In [16]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import math
import numpy as np

# RNN (from GoodFellow et al.)
class RNN(nn.Module):
    def __init__(self, emb_size, hidden_size, vocab_size, dropout=0.1):
        super(RNN, self).__init__()

        self.vocab_size = vocab_size

        # Embedding matrix
        # Lookup table from ids to trainable real value vectors embeddings
        self.embedding = nn.Linear(vocab_size, emb_size)

        # Input x_t to hidden h_t matrix
        self.W_in = nn.Linear(emb_size, hidden_size, bias=False)
        # Hidden h_{t-1} to hidden h_t matrix
        self.W_h = nn.Linear(hidden_size, hidden_size)
        # Hidden h_t to output y_t matrix
        self.W_out = nn.Linear(hidden_size, vocab_size)

        # Activation function tanh
        self.activation = nn.Tanh()

    # Forward pass
    def forward(self, prev_hidden, token):

        # Computes the embedding of the input token x_t
        input_emb = self.embedding(token)

        # Computes hidden state
        # h_t = a(W_in*x_t + W_h*h_{t-1} + b)
        hidden_state = self.activation(self.W_in(input_emb) + self.W_h(prev_hidden))

        # Computes unnormalized probabilities (logits) of next token y_t
        # y_t = W_out*h_t + b_out
        output = self.W_out(hidden_state)

        # NB! NO softmax beacuse PyTorch does it for you computing the Cross-Entropy Loss

        return output, hidden_state

In [18]:
# PyTorch RNN model
class LM_RNN(nn.Module):
    def __init__(self, emb_size, hidden_size, output_size, pad_index=0, n_layers=1):
        super(LM_RNN, self).__init__()
        # Lookup table from ids to trainable real value vectors embeddings
        self.embedding = nn.Embedding(output_size, emb_size, padding_idx=pad_index)
        # RNN torch layer
        self.rnn_type = 'rnn'
        self.rnn = nn.RNN(emb_size, hidden_size, n_layers, bidirectional=False, batch_first=True)
        self.pad_token = pad_index
        # Linear layer to project the hidden layer to output space (logits)
        self.output = nn.Linear(hidden_size, output_size)

    # Encoder pass
    def forward(self, input_sequence):
        # From tokens ids to real value embeddings
        # (batch_size, len(input_sequence)) -> (batch_size, len(input_sequence), emb_size)
        emb = self.embedding(input_sequence)
        # Compute representation for output using input x, hidden state h_{t-1}
        # (batch_size, len(input_sequence), emb_size) -> (batch_size , len(input_sequence) ,hidden_size*{2,1})
        rnn_out, last_hidden  = self.rnn(emb)
        # Classification logits of next token x_(t+1)
        # (batch_size, len(input_sequence), hidden_size*{2,1}) -> (batch_size, output_size, len(input_sequence))
        output = self.output(rnn_out).permute(0,2,1)
        return output, last_hidden

If you are interested in what is implemented here you can give a look at this paper from ICLR 2018:
https://openreview.net/pdf?id=SyyGPP0TZ

In [19]:
# Variational dropout
class VariationalDropout(nn.Module):
    def __init__(self, p=0.5):
        super().__init__()
        self.drop_p = p

    def forward(self, x):

        # If the network is in evaluation mode no dropout mask needed
        if not self.training:
            return x
        # Sample a single mask for every sequence from a bernulli distribution (batch_size, 1, size)
        mask = torch.empty(x.size(0), 1 , x.size(2), requires_grad=False).bernoulli_(1 - self.drop_p).to(DEVICE)
        # Expand same mask for every token in the sequence
        mask = mask.expand_as(x)
        # Apply the mask to the input tensor, scaling values accordingly to dropout probabilities
        drop_x = (mask * x) / (1 - self.drop_p)

        return drop_x

# LSTM + dropout model + AdamW optimization + weight tying + variational dropout
class LM_LSTM(nn.Module):
    def __init__(self, emb_size, hidden_size, output_size, pad_index=0, emb_drop_p=0.1, hidden_drop_p=0.1, n_layers=1):
        super(LM_LSTM, self).__init__()
        # Lookup table from ids to trainable real value vectors embeddings
        self.embedding = nn.Embedding(output_size, emb_size, padding_idx=pad_index)
        # Embedding dropout
        self.emb_dropout = VariationalDropout(emb_drop_p)
        # LSTM torch layer
        self.rnn_type = 'lstm'
        self.lstm = nn.LSTM(emb_size, hidden_size, n_layers, bidirectional=False, batch_first=True)
        self.pad_token = pad_index
        # Hidden layer dropout
        self.out_dropout = VariationalDropout(hidden_drop_p)
        # Linear layer to project the hidden layer to output space (logits)
        self.output = nn.Linear(hidden_size, output_size)
        # Weight tying between embedding layer (output_size, emb_size) and output layer (hidden_size, output_size)
        # !Note that emb_size = hidden_size and output_size = vocab_size
        # so (output_size, emb_size) = (hidden_size, output_size)^T
        self.output.weight = self.embedding.weight


    def forward(self, input_sequence):
        # From tokens ids to real value embeddings
        # (batch_size, len(input_sequence)) -> (batch_size, len(input_sequence), emb_size)
        emb = self.embedding(input_sequence)
        drop_emb = self.emb_dropout(emb)
        # Compute representation for output using input x, hidden state h_{t-1}
        # (batch_size, len(input_sequence), emb_size) -> (batch_size , len(input_sequence) ,hidden_size*{2,1})
        rnn_out, last_hidden  = self.lstm(drop_emb)
        drop_rnn_out = self.out_dropout(rnn_out)
        # Classification logits of next token x_{t+1}
        # (batch_size, len(input_sequence), hidden_size*{2,1}) -> (batch_size, output_size, len(input_sequence))
        output = self.output(drop_rnn_out).permute(0,2,1)
        return output, last_hidden

In [20]:
# Initialize weights of NN
def init_weights(mat):
    for m in mat.modules():
        if type(m) in [nn.GRU, nn.LSTM, nn.RNN]:
            for name, param in m.named_parameters():
                if 'weight_ih' in name:
                    for idx in range(4):
                        mul = param.shape[0]//4
                        torch.nn.init.xavier_uniform_(param[idx*mul:(idx+1)*mul])
                elif 'weight_hh' in name:
                    for idx in range(4):
                        mul = param.shape[0]//4
                        torch.nn.init.orthogonal_(param[idx*mul:(idx+1)*mul])
                elif 'bias' in name:
                    param.data.fill_(0)
        else:
            if type(m) in [nn.Linear]:
                torch.nn.init.uniform_(m.weight, -0.01, 0.01)
                if m.bias != None:
                    m.bias.data.fill_(0.01)

## 3. Train and validate the LM

Here is where the model learns via gradient descent his parameters $\boldsymbol{\theta^*}$ minimizing the Cross-Entropy loss over the training data ($\mathcal{D}_{train}$):

$$\boldsymbol{\theta^*} = argmin_{\boldsymbol{\theta}} L(\mathcal{D}_{train},\boldsymbol{\theta})$$

this gives us a model of the probability of the next token given the observed context, we can then use the model to generate text: $$P(w_{t+1}|(w_1,\dots,w_{t})) \approxeq P(y_t|\mathbf{h}_t,\mathbf{x}_t)$$

The goodness of the model is evaluated via the Perplexity measure: $PPL(W) = 2^{H(W)}$.

In [24]:
import torch.optim as optim

# Select the model you want to train (LSTM or RNN)
MODEL_TYPE = 'RNN'
# Careful if you use my model "emb_size" MUST BE EQUAL to "hidden_size" !!!

# emd_size e hidden_size possono cambiare per ingrandire modello e farlo funzionare meglio
HYPERPARAM = {
    "emb_size": 150,
    "hidden_size": 150,
    "n_epochs": 100,
    "patience": 5,
    "lr": 0.001,
    "beta_1": 0.9,
    "beta_2": 0.99,
    "eps": 1e-08,
    "weight_decay": 0,
    "clip": 5,
    "emb_drop_p": 0.2,
    "hidden_drop_p": 0.2
}

if MODEL_TYPE == 'LSTM':
    model = LM_LSTM(HYPERPARAM['emb_size'],
                    HYPERPARAM['hidden_size'],
                    len(lang.word2id),
                    pad_index=lang.word2id["<pad>"],
                    emb_drop_p=HYPERPARAM["emb_drop_p"],
                    hidden_drop_p=HYPERPARAM["hidden_drop_p"]).to(DEVICE)
elif MODEL_TYPE == 'RNN':
    model = LM_RNN(HYPERPARAM['emb_size'], HYPERPARAM['hidden_size'], len(lang.word2id)).to(DEVICE)

# Init the weights
model.apply(init_weights)
print(model)

# Define optimizer
optimizer = optim.AdamW(model.parameters(),
                        lr=HYPERPARAM['lr'],
                        betas=(HYPERPARAM['beta_1'],HYPERPARAM['beta_2']),
                        eps=HYPERPARAM['eps'],
                        weight_decay=HYPERPARAM['weight_decay'])
# Define Cross-Entropy loss
train_loss = nn.CrossEntropyLoss(ignore_index=lang.word2id["<pad>"])
eval_loss = nn.CrossEntropyLoss(ignore_index=lang.word2id["<pad>"], reduction='sum')

LM_RNN(
  (embedding): Embedding(10001, 150, padding_idx=0)
  (rnn): RNN(150, 150, batch_first=True)
  (output): Linear(in_features=150, out_features=10001, bias=True)
)


In [22]:
# Train epoch
def train_loop(data, optimizer, criterion, model, clip=5):

    # Model in training mode
    model.train()

    # Loss
    loss_array = []
    number_of_tokens = []

    # Iterate over batches
    for sample in data:

        # Zeroing the gradient
        optimizer.zero_grad()

        # Model inference
        output, _ = model(sample['source'])

        # Compute batch loss
        loss = criterion(output, sample['target'])
        loss_array.append(loss.item() * sample["number_tokens"])
        number_of_tokens.append(sample["number_tokens"])

        # Compute the gradient, deleting the computational graph
        loss.backward()
        # Clip the gradient to avoid explosioning gradients (RNN backprop issue)
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)

        # Backprop step, update the weights via the optimizer
        optimizer.step()

    return sum(loss_array)/sum(number_of_tokens)

# Model test/evaluation
def eval_loop(data, eval_criterion, model):

    # Model in evaluation mode
    model.eval()

    # Loss
    loss_to_return = []
    loss_array = []
    number_of_tokens = []

    # Disable computational graph
    with torch.no_grad():
        # Iterate over batches
        for sample in data:

            # Model inference
            output, _ = model(sample['source'])

            # Compute batch loss
            loss = eval_criterion(output, sample['target'])
            loss_array.append(loss.item())
            number_of_tokens.append(sample["number_tokens"])

    # Compute perplexity and average loss
    ppl = math.exp(sum(loss_array) / sum(number_of_tokens))
    loss_to_return = sum(loss_array) / sum(number_of_tokens)
    return ppl, loss_to_return

Finally train the model!

In [23]:
import math
from tqdm import tqdm
import copy

# Training hyperparameters
n_epochs = HYPERPARAM['n_epochs']
patience = HYPERPARAM['patience']

# Losses
losses_train = []
losses_dev = []
sampled_epochs = []
best_ppl = math.inf

# Model checkpoints
best_model = None

# TRAIN

pbar = tqdm(range(1,n_epochs))
# Iterate over epochs
for epoch in pbar:

    # Epoch training
    loss = train_loop(train_loader, optimizer, train_loss, model, HYPERPARAM['clip'])

    # Evaluation on sampled epochs
    if epoch % 1 == 0:

        # Compute loss on evaluation set
        sampled_epochs.append(epoch)
        losses_train.append(np.asarray(loss).mean())
        ppl_dev, loss_dev = eval_loop(dev_loader, eval_loss, model)
        losses_dev.append(np.asarray(loss_dev).mean())
        pbar.set_description(f'(TRAIN) - CrossEntropy: {loss}. (DEV) - PPL: {ppl_dev}, CrossEntropy: {loss_dev}')

        # If the model is better copy it and reset patience
        # Otherwise diminuish patience level for early stopping
        if  ppl_dev < best_ppl:
            best_ppl = ppl_dev
            best_model = copy.deepcopy(model).to('cpu')
            patience = 3
        else:
            patience -= 1

        # Early stopping with patience
        if patience <= 0:
            break

# TEST

best_model.to(DEVICE)
final_ppl,  final_loss = eval_loop(test_loader, eval_loss, best_model)
print(f'(TEST) - PPL: {final_ppl}, CrossEntropy: {final_loss}')

(TRAIN) - CrossEntropy: 4.310047541172273. (DEV) - PPL: 160.84716025787387, CrossEntropy: 5.080454598930956:  11%|█         | 11/99 [03:13<25:50, 17.62s/it]


(TEST) - PPL: 148.40480941041898, CrossEntropy: 4.999943738634142


If you want to save your model and use it later remeber to download it to your laptop because Colab will reset the session everytime!

In [ ]:
MODEL_PATH = 'model.pt'
HYPERPARAM_PATH = 'hyperparams.json'

In [ ]:
import json

# Save model
torch.save(model.state_dict(), MODEL_PATH)
# Save hyperparameters
with open(HYPERPARAM_PATH, "w") as outfile:
      json.dump(HYPERPARAM, outfile)

You can later load it and use it uploading to Colab the 'model_name.pt' and 'hyperparams.json' files!

In [ ]:
import json

# Select the model you want to load
MODEL_TYPE = 'LSTM'

# Load hyperparameters
with open(HYPERPARAM_PATH, "r") as infile:
    HYPERPARAM = json.load(infile)

# To load the model you need to initialize it (with the same hyperparameters)
if MODEL_TYPE == 'RNN':
    model = LM_RNN(HYPERPARAM['emb_size'], HYPERPARAM['hidden_size'], len(lang.word2id)).to(DEVICE)
elif MODEL_TYPE == 'LSTM':
    model = LM_LSTM(HYPERPARAM['emb_size'],
                HYPERPARAM['hidden_size'],
                len(lang.word2id),
                pad_index=lang.word2id["<pad>"],
                emb_drop_p=HYPERPARAM["emb_drop_p"],
                hidden_drop_p=HYPERPARAM["hidden_drop_p"]).to(DEVICE)

# Then you load it
model_weights = torch.load(MODEL_PATH, map_location=DEVICE)
model.load_state_dict(model_weights)

<ipython-input-45-d0ad1b2e1fd4>:22: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_weights = torch.load(MODEL_PATH, map_location=DEVICE)


<All keys matched successfully>

## 4. Generate Text

It was a long run but we finally have our LM and we can use it to generating text. Here I implemented for you an easy technique called Top-K sampling, just provide the prompt (context) and the LM will generate text autoregressively as you usually do with ChatGPT (if you use it).

We are not OpenAI so our model is way worse, as for ChatGPT you should be careful of which test it produce!

In [ ]:
import random

class TopKDecoder(nn.Module):
    def __init__(self, decoder, tokens_to_ignore_ids, k=5):
        super(TopKDecoder, self).__init__()
        # Decoder model
        self.decoder = decoder
        self.softmax = nn.Softmax(dim=2)
        # K value
        self.k = k
        # Ids of tokens that won't be generated
        self.tokens_to_ignore_ids = tokens_to_ignore_ids

    # Decoder single pass
    def forward(self, last_hidden):
        # Model in evaluation mode
        self.decoder.eval()
        # Discard cell state for LSTM
        if self.decoder.rnn_type == 'lstm':
          lstm_last_hidden = last_hidden
          last_hidden = last_hidden[0]
        # Get final hidden layer (h_t)
        # (1, n_layers*{2,1}, hidden_size) -> (1, 1, hidden_size)
        final_last_hidden = last_hidden[-1:]
        # Generate the next token likelihood (y_t)
        # (1, 1, hidden_size) -> (1, 1, output_size)
        output = self.softmax(model.output(final_last_hidden))
        # Remove ignore tokens from counting
        for id in self.tokens_to_ignore_ids:
          output[0,0,id] = -1
        # Get the top-K tokens
        top_k_tokens = self.get_top_k_tokens(output)
        # Sample
        chosen_token = self.weighted_sampling(top_k_tokens)
        # Generate next hidden state (h_{t+1})
        input = torch.LongTensor(1,1).to(DEVICE).detach()
        input[0,0] = chosen_token
        emb = model.embedding(input)
        if self.decoder.rnn_type == 'rnn':
            _, current_hidden  = model.rnn(emb, last_hidden)
        elif self.decoder.rnn_type == 'lstm':
            _, current_hidden  = model.lstm(emb, lstm_last_hidden)
        return chosen_token, current_hidden

    # Extract the top-K tokens
    def get_top_k_tokens(self, output):
        top_k_tokens = []
        for i in range(0,self.k):
            # Pick current best token (id, likelihood)
            max_index = torch.argmax(output)
            top_k_tokens.append((max_index, torch.max(output)))
            # Remove current best from counting
            output[0,0,max_index] = -1
        return top_k_tokens

    # Weighted random sampling
    def weighted_sampling(self, x):
        value = random.uniform(0,sum([likelihood for (_,likelihood) in x]))
        counter = 0
        for token_id, likelihood in x:
            if value <= (counter + likelihood):
                return token_id
            counter += likelihood

In [ ]:
# Generation parameters
TOKENS_TO_IGNORE = ['<unk>','<pad>']
K = 3

# Autoregressively generate text using the top-K sampling technique
def autoregressive_generation(encoder, decoder, prompt, eos_id, max_length=50):

    # Model in evaluation mode
    encoder.eval()
    # Encode the prompt
    _, last_hidden = encoder(prompt)

    # Generate at most max_length tokens
    generated_text = []
    for i in range(0,max_length):
        # Single step generation
        chosen_token, last_hidden = decoder(last_hidden)
        # If <eos> stop generating
        if chosen_token.item() == eos_id:
          break
        generated_text.append(chosen_token.item())

    return generated_text

def generate_text(model, prompt):
    # Convert text prompt into Torch tensors
    prompt_converter = PennTreeBank([], lang)
    batched_prompt = torch.LongTensor(1,len(prompt.split())).to(DEVICE).detach()
    batched_prompt[0:] = torch.LongTensor(prompt_converter.mapping_seq([prompt.split()], lang)[0]).to(DEVICE).detach()

    # Instantiate decoder
    tokens_to_ignore_ids = [lang.word2id[token] for token in TOKENS_TO_IGNORE]
    decoder = TopKDecoder(model, tokens_to_ignore_ids, K)

    # Autoregressive generation
    model_reply = autoregressive_generation(model, decoder, batched_prompt, lang.word2id['<eos>'], max_length=20)

    # Convert back ids into tokens
    reply = "".join([lang.id2word[id]+" " for id in model_reply])
    return reply

In [ ]:
# Prompt (context)
# Words are splitted by whitespace and must be in the vocabulary (only lowercase tokens) !!!
PROMPT = 'i was saying that'

# To check tokenization ad words availability
#print(PROMPT.split())
#print(lang.word2id['i'])

# Generate text using the LM
reply = generate_text(model, PROMPT)

print(f'PROMPT: {PROMPT}')
print(f'LM: {reply}')

PROMPT: i was saying that
LM: mr. bush 's veto is the first time to have a new position 
